In [1]:
import torch, torchvision

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
cuda available: True
gpu: Tesla T4


In [2]:
!pip -q install "transformers==4.37.2" "tokenizers==0.15.1" "timm==0.9.12" "sentencepiece==0.1.99" einops accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 60.6 MB/s eta 0:00:00:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 95.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 68.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.37.2 which is incompatible.


In [3]:
import torch, torchvision, transformers, timm, tokenizers, sentencepiece

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("transformers:", transformers.__version__)
print("timm:", timm.__version__)
print("tokenizers:", tokenizers.__version__)
print("sentencepiece:", sentencepiece.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
transformers: 4.37.2
timm: 0.9.12
tokenizers: 0.15.1
sentencepiece: 0.1.99
cuda available: True
gpu: Tesla T4


In [4]:
import os
import re
import gc
import json
from pathlib import Path

from PIL import Image
import torch
from torchvision import transforms as T
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoTokenizer, AutoModel

# ========= 你需要改的地方 =========
IMAGE_DIR = Path("/kaggle/input/datasets/yejixuan/real-dataset")
MODEL_NAME = "OpenGVLab/InternVL2_5-2B"
NUM_OBJECTS = 8
LONG_SIDE = 1024
INPUT_SIZE = 448
# =================================

WORK_DIR = Path("/kaggle/working/internvl_perception")
WORK_DIR.mkdir(parents=True, exist_ok=True)

RESIZED_DIR = WORK_DIR / "resized_1024"
RESIZED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSONL = WORK_DIR / "perception_results.jsonl"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("DEVICE:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("IMAGE_DIR exists:", IMAGE_DIR.exists())
print("WORK_DIR:", WORK_DIR)

DEVICE: cuda
GPU: Tesla T4
IMAGE_DIR exists: True
WORK_DIR: /kaggle/working/internvl_perception


In [5]:
from pathlib import Path

VALID_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

image_paths = sorted([
    p for p in IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in VALID_EXTS
])

print(f"Found {len(image_paths)} images")
for p in image_paths[:5]:
    print(p.name)

if len(image_paths) == 0:
    raise ValueError(f"No images found in {IMAGE_DIR}")

Found 314 images
85013611_p0_master1200.jpg
85027430_p0_master1200.jpg
85033409_p0_master1200.jpg
85053658_p0_master1200.jpg
85054567_p0_master1200.jpg


In [6]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=False
)

print("Loading model...")
model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    trust_remote_code=True
).eval().to(DEVICE)

print("Model loaded successfully.")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_internlm2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/InternVL2_5-2B:
- tokenization_internlm2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

FlashAttention2 is not installed.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/4.41G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Model loaded successfully.


In [7]:
# 图像预处理
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

transform = T.Compose([
    T.Lambda(lambda img: img.convert("RGB")),
    T.Resize((INPUT_SIZE, INPUT_SIZE), interpolation=InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def resize_long_side_pil(image, long_side=1024):
    w, h = image.size
    if max(w, h) == long_side:
        return image

    if w >= h:
        new_w = long_side
        new_h = int(h * long_side / w)
    else:
        new_h = long_side
        new_w = int(w * long_side / h)

    return image.resize((new_w, new_h), Image.BICUBIC)

def prepare_image(image_path):
    """
    1) 读图
    2) 长边 resize 到 1024
    3) 保存 1024 版本
    4) 转成 InternVL 输入 tensor: [1, 3, 448, 448]
    """
    image = Image.open(image_path).convert("RGB")
    image_1024 = resize_long_side_pil(image, long_side=LONG_SIDE)

    save_path = RESIZED_DIR / image_path.name
    image_1024.save(save_path)

    pixel_values = transform(image_1024).unsqueeze(0)   # [1, 3, H, W]
    return image_1024, save_path, pixel_values

In [8]:
# prompt 和输出清洗（适配后续 segmentation）
CAPTION_PROMPT = """Describe this image clearly, shortly and precisely.
Use no more than 40 words.
This description should be helpful for restoring the image.
Ignore text in the image.
Only output one sentence."""

def build_object_prompt(caption, n=8):
    return f"""Based on the image and its description, list exactly {n} visually localizable objects in the image.
Focus on concrete objects, clothing items, accessories, animals, props, or clearly visible body parts that can be segmented.
Each item should be a short English noun phrase of 1 to 3 words.
Avoid background, text, year, style, mood, action, abstract concepts, and generic words like character, person, image.
Image description: {caption}
Only output a comma-separated list."""

DROP_EXACT = {
    "image", "picture", "scene", "style", "anime", "art", "drawing",
    "illustration", "background", "foreground", "view", "setting",
    "light", "lighting", "mood", "quality", "text", "character",
    "person", "people", "year", "new year", "word", "words", "letter",
    "letters", "pattern"
}

DROP_IF_CONTAINS = {
    "background", "foreground", "style", "mood", "quality",
    "new year", "happy new year", "text", "letter", "word"
}

SPELL_FIX = {
    "kimonono": "kimono"
}

PHRASE_REWRITE = {
    "knee-high socks": "socks",
    "knee high socks": "socks",
    "hair ornament": "hairpin",
    "hair accessory": "hairpin",
}

PRIORITY_HEADS = [
    "hairpin", "bracelet", "sandals", "socks", "kimono", "robe",
    "scroll", "mouse", "plate", "flower", "fan", "ribbon",
    "bow", "cat", "hat", "hair", "braid"
]

def singularize_token(token):
    if token.endswith("ies") and len(token) > 4:
        return token[:-3] + "y"
    if token.endswith("s") and len(token) > 3 and not token.endswith(("ss", "us", "is")):
        return token[:-1]
    return token

def normalize_phrase(p):
    p = p.lower().strip()
    p = p.replace("_", " ")
    p = re.sub(r"[\(\)\[\]\{\}\"'`]", " ", p)
    p = re.sub(r"[^a-zA-Z0-9\- ]", " ", p)
    p = re.sub(r"\s+", " ", p).strip()

    p = SPELL_FIX.get(p, p)
    p = PHRASE_REWRITE.get(p, p)

    tokens = []
    for t in p.split():
        if t.isdigit():
            continue
        t = SPELL_FIX.get(t, t)
        t = singularize_token(t)
        tokens.append(t)

    p = " ".join(tokens).strip()
    p = PHRASE_REWRITE.get(p, p)
    return p

def should_drop_phrase(p):
    if not p:
        return True
    if p in DROP_EXACT:
        return True
    for bad in DROP_IF_CONTAINS:
        if bad in p:
            return True
    if len(p.split()) > 3:
        return True
    return False

def phrase_to_seg_label(p):
    tokens = p.split()
    for head in PRIORITY_HEADS:
        if head in tokens:
            return head
    return tokens[-1]

def clean_object_phrases(text, n=8):
    text = text.strip().lower().replace("\n", ",")
    parts = re.split(r"[,;，；]\s*", text)

    cleaned = []
    seen = set()

    for p in parts:
        p = re.sub(r"^\d+[\.\)\-: ]*", "", p).strip()
        p = normalize_phrase(p)

        if should_drop_phrase(p):
            continue
        if p in seen:
            continue

        seen.add(p)
        cleaned.append(p)

        if len(cleaned) >= n:
            break

    return cleaned

In [9]:
# 调用 InternVL 生成 caption 和 object phrases / object words
@torch.inference_mode()
def ask_image(prompt, pixel_values, max_new_tokens=80):
    pixel_values = pixel_values.to(device=DEVICE, dtype=DTYPE)

    generation_config = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )

    question = "<image>\n" + prompt.strip()

    response = model.chat(
        tokenizer,
        pixel_values,
        question,
        generation_config
    )
    return response.strip()

def generate_caption_and_objects(image_path, num_objects=8):
    image_1024, save_path, pixel_values = prepare_image(image_path)

    caption = ask_image(CAPTION_PROMPT, pixel_values, max_new_tokens=80)

    object_prompt = build_object_prompt(caption, n=num_objects)
    raw_objects = ask_image(object_prompt, pixel_values, max_new_tokens=64)

    object_phrases = clean_object_phrases(raw_objects, n=num_objects)
    object_words = [phrase_to_seg_label(x) for x in object_phrases]

    result = {
        "image_path": str(image_path),
        "resized_path": str(save_path),
        "resized_width": image_1024.size[0],
        "resized_height": image_1024.size[1],
        "caption": caption,
        "raw_object_output": raw_objects,
        "object_phrases": object_phrases,   # 给 segmentation 用
        "object_words": object_words,       # 兼容旧字段
    }

    del pixel_values
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return result

In [10]:
results = []

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for idx, img_path in enumerate(image_paths, start=1):
        print(f"[{idx}/{len(image_paths)}] {img_path.name}")

        try:
            result = generate_caption_and_objects(img_path, num_objects=NUM_OBJECTS)
            results.append(result)
            f.write(json.dumps(result, ensure_ascii=False) + "\n")

        except torch.cuda.OutOfMemoryError as e:
            err = {
                "image_path": str(img_path),
                "error": f"OOM: {str(e)}"
            }
            results.append(err)
            f.write(json.dumps(err, ensure_ascii=False) + "\n")

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            print("OOM on:", img_path.name)

        except Exception as e:
            err = {
                "image_path": str(img_path),
                "error": str(e)
            }
            results.append(err)
            f.write(json.dumps(err, ensure_ascii=False) + "\n")
            print("ERROR on:", img_path.name, "|", e)

print("Done.")
print("Saved to:", OUTPUT_JSONL)

[1/314] 85013611_p0_master1200.jpg
[2/314] 85027430_p0_master1200.jpg
[3/314] 85033409_p0_master1200.jpg
[4/314] 85053658_p0_master1200.jpg
[5/314] 85054567_p0_master1200.jpg
[6/314] 85055484_p0_master1200.jpg
[7/314] 85067355_p0_master1200.jpg
[8/314] 85076069_p0_master1200.jpg
[9/314] 85077638_p0_master1200.jpg
[10/314] 85095880_p0_master1200.jpg
[11/314] 85097516_p0_master1200.jpg
[12/314] 85097641_p0_master1200.jpg
[13/314] 85099021_p0_master1200.jpg
[14/314] 85101470_p0_master1200.jpg
[15/314] 85113914_p0_master1200.jpg
[16/314] 85116358_p0_master1200.jpg
[17/314] 85117473_p0_master1200.jpg
[18/314] 85126056_p0_master1200.jpg
[19/314] 85129001_p0_master1200.jpg
[20/314] 85141923_p0_master1200.jpg
[21/314] 85144321_p0_master1200.jpg
[22/314] 85148467_p0_master1200.jpg
[23/314] 85156638_p0_master1200.jpg
[24/314] 85157814_p0_master1200.jpg
[25/314] 85159715_p0_master1200.jpg
[26/314] 85168631_p0_master1200.jpg
[27/314] 85183553_p0_master1200.jpg
[28/314] 85185777_p0_master1200.jpg
[